# RIN - Experiment 6: topology-aware retrieval correction

This notebook keeps the same golden micro-world and the same query from experiments 3-4, then asks two narrower questions.

1. If experiment 4 is right that the failure is about convergence topology, does topology surgery alone flip the answer?
2. On the unchanged original graph, which topology-aware correction, if any, is strong enough to make `caffeine` outrank `beverage`?


In [1]:
import sys
from pathlib import Path

def find_repo_root(marker="requirements.txt"):
    path = Path.cwd()
    for candidate in [path, *path.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"could not find repo root (looked for {marker})")

REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


In [2]:
%load_ext autoreload
%autoreload 2


In [3]:
from collections import defaultdict

from embryonic.numberbatch import Numberbatch
from embryonic.query import query_vectors
from memory.diffuse import diffuse
from memory.edge import Edge
from memory.graph import MemoryGraph
from memory.node import Node
from memory.retrieval import nearest_nodes_by_embedding
from memory.similarity import cosine_similarity

NUMBERBATCH_FILE = REPO_ROOT / "source" / "numberbatch-en-19.08.txt.gz"

NODE_SPECS = [
    ("coffee", "/c/en/coffee", "concept", 0.7),
    ("caffeine", "/c/en/caffeine", "concept", 0.7),
    ("beverage", "/c/en/beverage", "concept", 0.7),
    ("hot drink", "/c/en/hot_drink", "concept", 0.7),
    ("tea", "/c/en/tea", "concept", 0.7),
    ("juice", "/c/en/juice", "concept", 0.7),
    ("soda", "/c/en/soda", "concept", 0.7),
    ("north dakota", "/c/en/north_dakota", "concept", 0.7),
    ("south dakota", "/c/en/south_dakota", "concept", 0.7),
    ("north", "/c/en/north", "concept", 0.7),
]

BASE_EDGE_SPECS = [
    ("coffee", "caffeine", "HasProperty", 1.0),
    ("coffee", "beverage", "IsA", 1.0),
    ("coffee", "hot drink", "RelatedTo", 1.0),
    ("hot drink", "beverage", "IsA", 1.0),
    ("beverage", "tea", "RelatedTo", 1.0),
    ("beverage", "juice", "RelatedTo", 1.0),
    ("beverage", "soda", "RelatedTo", 1.0),
    ("north dakota", "south dakota", "RelatedTo", 1.0),
]

QUERY = "why coffee gives energy"

def build_graph(numberbatch, remove_edges=None, add_edges=None):
    remove_edges = set(remove_edges or [])
    G = MemoryGraph()
    content_to_id = {}
    for i, (content, uri, kind, conf) in enumerate(NODE_SPECS):
        vector = numberbatch.vector_for(uri)
        assert vector is not None, f"no vector for {uri}"
        G.add_node(Node(id=i, content=content, embedding=vector, node_kind=kind,
                        confidence=conf, provenance=[uri]))
        content_to_id[content] = i
    edge_specs = list(BASE_EDGE_SPECS)
    if add_edges:
        edge_specs.extend(add_edges)
    for src, dst, etype, weight in edge_specs:
        if (src, dst, etype, weight) in remove_edges:
            continue
        G.add_edge(Edge(src=content_to_id[src], dst=content_to_id[dst], type=etype, weight=weight))
        G.add_edge(Edge(src=content_to_id[dst], dst=content_to_id[src], type=etype, weight=weight))
    return G, content_to_id

def anchors_for_query(G, query_vecs, top_k=3):
    anchors = set()
    for vec in query_vecs:
        anchors.update(nearest_nodes_by_embedding(G, vec, top_k=top_k))
    return anchors

def diffuse_with_post_accumulation_penalty(
    G,
    source_amplitudes,
    query_vecs=None,
    degree_gamma=0.0,
    active_parent_gamma=0.0,
    decay=0.8,
    min_propagate_fraction=0.01,
    max_steps=20,
):
    relevance_cache = {}

    def relevance(node_id):
        if query_vecs is None:
            return 1.0
        if node_id not in relevance_cache:
            node = G.nodes[node_id]
            if node.embedding is None:
                relevance_cache[node_id] = 1.0
            else:
                best = max(cosine_similarity(node.embedding, qv) for qv in query_vecs)
                relevance_cache[node_id] = max(0.0, best)
        return relevance_cache[node_id]

    floor = min_propagate_fraction * max(source_amplitudes.values(), default=0.0)
    current = dict(source_amplitudes)
    peak = dict(current)

    for _step in range(max_steps):
        active = {node_id: amount for node_id, amount in current.items() if amount >= floor}
        if not active:
            break

        incoming = defaultdict(float)
        active_parents = defaultdict(set)
        for node_id, amount in active.items():
            out_edges = G.out_edges(node_id)
            gated_weights = [edge.weight * relevance(edge.dst) for edge in out_edges]
            total_weight = sum(gated_weights)
            if total_weight <= 0:
                continue
            for edge, gated_weight in zip(out_edges, gated_weights):
                incoming[edge.dst] += decay * amount * (gated_weight / total_weight)
                if gated_weight > 0:
                    active_parents[edge.dst].add(node_id)

        current = {}
        for dst, amount in incoming.items():
            adjusted = amount
            if degree_gamma > 0:
                adjusted /= max(1, G.degree(dst)) ** degree_gamma
            if active_parent_gamma > 0:
                adjusted /= max(1, len(active_parents[dst])) ** active_parent_gamma
            current[dst] = adjusted

        for node_id, amount in current.items():
            if amount > peak.get(node_id, 0.0):
                peak[node_id] = amount
    return peak

def print_pair_score(label, scores, content_to_id):
    caffeine = scores[content_to_id["caffeine"]]
    beverage = scores[content_to_id["beverage"]]
    print(f"{label:42s} caffeine={caffeine:6.2f} beverage={beverage:6.2f} win={caffeine > beverage}")


In [4]:
numberbatch = Numberbatch()
numberbatch.load(str(NUMBERBATCH_FILE))
print(f"loaded {len(numberbatch.terms):,} terms")

query_vecs = query_vectors(QUERY, numberbatch)
print("query:", QUERY)


loaded 516,782 terms
query: why coffee gives energy


## Part A: topology surgery controls

Change only graph connectivity; keep the query, embeddings, and semantic gate fixed.


In [5]:
topology_variants = [
    ("original graph", [], []),
    (
        "remove coffee-beverage and beverage-tea",
        [("coffee", "beverage", "IsA", 1.0), ("beverage", "tea", "RelatedTo", 1.0)],
        [],
    ),
    (
        "add hot drink-caffeine and tea-caffeine",
        [],
        [("hot drink", "caffeine", "RelatedTo", 1.0), ("tea", "caffeine", "RelatedTo", 1.0)],
    ),
    (
        "both modifications at once",
        [("coffee", "beverage", "IsA", 1.0), ("beverage", "tea", "RelatedTo", 1.0)],
        [("hot drink", "caffeine", "RelatedTo", 1.0), ("tea", "caffeine", "RelatedTo", 1.0)],
    ),
]

for label, remove_edges, add_edges in topology_variants:
    G, content_to_id = build_graph(numberbatch, remove_edges=remove_edges, add_edges=add_edges)
    anchors = anchors_for_query(G, query_vecs)
    source_amplitudes = {anchor: 100.0 for anchor in anchors}
    scores = diffuse(G, source_amplitudes, decay=0.8, min_propagate_fraction=0.01,
                     max_steps=20, query_vectors=query_vecs)
    print_pair_score(label, scores, content_to_id)
    print("  anchors:", [G.nodes[a].content for a in sorted(anchors)])


original graph                             caffeine= 27.20 beverage=129.46 win=False
  anchors: ['coffee', 'hot drink', 'tea']
remove coffee-beverage and beverage-tea    caffeine= 37.57 beverage= 27.38 win=True
  anchors: ['coffee', 'hot drink', 'tea']
add hot drink-caffeine and tea-caffeine    caffeine= 95.10 beverage= 77.17 win=True
  anchors: ['coffee', 'hot drink', 'tea']
both modifications at once                 caffeine=141.30 beverage= 19.26 win=True
  anchors: ['coffee', 'hot drink', 'tea']


## Part B: correction ablation on the original graph

Now restore the original graph and change only the retrieval rule.


In [6]:
G, content_to_id = build_graph(numberbatch)
anchors = anchors_for_query(G, query_vecs)
source_amplitudes = {anchor: 100.0 for anchor in anchors}

print("anchors:", [G.nodes[a].content for a in sorted(anchors)])
print()

variants = [
    ("gate only", dict()),
    ("divide by deg(B)^(1/2)", dict(degree_gamma=0.5)),
    ("divide by deg(B)", dict(degree_gamma=1.0)),
    ("divide by fanin_t(B)^(1/2)", dict(active_parent_gamma=0.5)),
    ("divide by fanin_t(B)", dict(active_parent_gamma=1.0)),
]

for label, kwargs in variants:
    if not kwargs:
        scores = diffuse(G, source_amplitudes, decay=0.8, min_propagate_fraction=0.01,
                         max_steps=20, query_vectors=query_vecs)
    else:
        scores = diffuse_with_post_accumulation_penalty(
            G,
            source_amplitudes,
            query_vecs=query_vecs,
            decay=0.8,
            min_propagate_fraction=0.01,
            max_steps=20,
            **kwargs,
        )
    print_pair_score(label, scores, content_to_id)


anchors: ['coffee', 'hot drink', 'tea']

gate only                                  caffeine= 27.20 beverage=129.46 win=False
divide by deg(B)^(1/2)                     caffeine= 27.20 beverage= 57.89 win=False
divide by deg(B)                           caffeine= 27.20 beverage= 25.89 win=True
divide by fanin_t(B)^(1/2)                 caffeine= 27.20 beverage= 74.74 win=False
divide by fanin_t(B)                       caffeine= 27.20 beverage= 43.15 win=False


## Regression check: `north dakota`

Re-run the already-correct multi-word fixture with the strongest candidate correction.


In [7]:
QUERY_REGRESSION = "tell me about north dakota"
regression_vecs = query_vectors(QUERY_REGRESSION, numberbatch)
anchors = anchors_for_query(G, regression_vecs)
source_amplitudes = {anchor: 100.0 for anchor in anchors}
scores = diffuse_with_post_accumulation_penalty(
    G,
    source_amplitudes,
    query_vecs=regression_vecs,
    degree_gamma=1.0,
    decay=0.8,
    min_propagate_fraction=0.01,
    max_steps=20,
)

print("anchors:", [G.nodes[a].content for a in sorted(anchors)])
print("top results:")
for node_id, amount in sorted(scores.items(), key=lambda kv: -kv[1])[:5]:
    print(f"  {G.nodes[node_id].content!r:15s} {amount:7.2f}")


anchors: ['north dakota', 'south dakota']
top results:
  'south dakota'   100.00
  'north dakota'   100.00
